In [6]:
import pandas as pd
import numpy as np
import joblib
# Load the raw hourly data
# We use ../ because the notebook is in the 'notebooks' folder
df_hourly = pd.read_csv("../data/weather_raw.csv")
df_hourly['date'] = pd.to_datetime(df_hourly['date'])

print(f"Loaded {len(df_hourly)} hourly rows.")

Loaded 298056 hourly rows.


In [7]:
# Group by day ('D') and aggregate
df_daily = df_hourly.resample('D', on='date').agg({
    'temperature': ['max', 'min', 'mean'],
    'humidity': 'mean',
    'pressure': 'mean',
    'precipitation': 'sum',
    'wind_speed': 'max'
})

# Flatten the multi-index columns (Pandas creates complex headers during agg)
df_daily.columns = [
    'temp_max', 'temp_min', 'temp_mean', 
    'humidity_mean', 'pressure_mean', 
    'precip_total', 'wind_max'
]
df_daily = df_daily.reset_index()

print(f"Resampled to {len(df_daily)} daily rows.")
df_daily.head()

Resampled to 12419 daily rows.


,date,temp_max,temp_min,temp_mean,humidity_mean,pressure_mean,precip_total,wind_max
0,1990-01-01,14.359500,11.209499,13.007416,79.698394,1014.754782,4.6,21.862406
1,1990-01-02,14.209499,8.559500,11.167833,86.329791,1010.640433,7.9,38.546906
2,1990-01-03,12.309500,9.109500,10.872000,76.518009,1007.523595,5.9,26.925497
3,1990-01-04,13.659499,11.009500,12.330333,84.490857,1017.292949,2.0,24.472057
4,1990-01-05,14.309500,11.859500,13.021999,89.013759,1020.468115,0.1,9.904906


In [8]:
# 1. Target: Temperatura Máxima de amanhã (O que queremos prever)
df_daily['target_next_day_max'] = df_daily['temp_max'].shift(-1)

# 2. Lag Features: Temperaturas de dias anteriores (Contexto temporal)
df_daily['temp_max_lag_1'] = df_daily['temp_max'].shift(1)
df_daily['temp_max_lag_2'] = df_daily['temp_max'].shift(2) # <--- Novo: há 2 dias
df_daily['temp_max_lag_3'] = df_daily['temp_max'].shift(3) # <--- Novo: há 3 dias

# 3. Média Móvel de 3 dias (Inércia térmica/Tendência)
df_daily['temp_max_mean_3d'] = df_daily['temp_max'].rolling(window=3).mean()

# 4. Amplitude Térmica (Diferença entre o pico de calor e o frio da noite)
df_daily['temp_range'] = df_daily['temp_max'] - df_daily['temp_min']

# 5. Seasonal Encoding (Meses em formato cíclico)
df_daily['month'] = df_daily['date'].dt.month
df_daily['sin_month'] = np.sin(2 * np.pi * df_daily['month'] / 12)
df_daily['cos_month'] = np.cos(2 * np.pi * df_daily['month'] / 12)

# REMOVER NAANs: O rolling(3) e o shift(3) criam 3 linhas vazias no início.
# O shift(-1) do target cria 1 linha vazia no fim.
df_daily = df_daily.dropna()

# Guardar os dados limpos
df_daily.to_csv("../data/weather_daily.csv", index=False)
print("Novas features calculadas e dataset guardado!")

Novas features calculadas e dataset guardado!


In [9]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# 1. Selection of Features
# We exclude 'date' because the model only understands numbers
# We exclude 'target_next_day_max' because that's what we want to predict
features = [
    'temp_max', 'temp_min', 'temp_mean', 'humidity_mean', 
    'pressure_mean', 'precip_total', 'wind_max', 
    'temp_max_lag_1', 'temp_max_lag_2', 'temp_max_lag_3', # Lags
    'temp_max_mean_3d', 'temp_range',                     # Novas métricas
    'sin_month', 'cos_month'
]

# 2. Scaling
scaler_features = MinMaxScaler(feature_range=(0, 1))
scaler_target = MinMaxScaler(feature_range=(0, 1))

# Scale features and target separately (so we can inverse scale the prediction later)
scaled_features = scaler_features.fit_transform(df_daily[features])
scaled_target = scaler_target.fit_transform(df_daily[['target_next_day_max']])
# Guardar os scalers para usar nos outros notebooks e na App
joblib.dump(scaler_features, '../models/scaler_features.joblib')
joblib.dump(scaler_target, '../models/scaler_target.joblib')

print("Scalers guardados com sucesso!")

# 3. Create Sequences (Sliding Window of 7 days)
def create_sequences(data, target, window_size=7):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:(i + window_size)])
        y.append(target[i + window_size])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_features, scaled_target, window_size=7)

print(f"Shape of X (Input): {X.shape}") # Should be (Samples, 7, 10)
print(f"Shape of y (Target): {y.shape}") # Should be (Samples, 1)

Scalers guardados com sucesso!
Shape of X (Input): (12408, 7, 14)
Shape of y (Target): (12408, 1)


In [10]:
# 1. Define split points
total_samples = len(X)
train_end = int(total_samples * 0.8)
val_end = int(total_samples * 0.9)

# 2. Slice the arrays (No shuffling!)
X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f"Training set:   {X_train.shape}") # ~2910 days
print(f"Validation set: {X_val.shape}")   # ~363 days
print(f"Test set:       {X_test.shape}")  # ~365 days (Full year 2024)

# 3. Save the preprocessed tensors
# We use .npy format for efficiency with NumPy arrays
np.save("../data/X_train.npy", X_train)
np.save("../data/y_train.npy", y_train)
np.save("../data/X_val.npy", X_val)
np.save("../data/y_val.npy", y_val)
np.save("../data/X_test.npy", X_test)
np.save("../data/y_test.npy", y_test)

print("\nAll tensors saved to the 'data' folder. Preprocessing complete!")

Training set:   (9926, 7, 14)
Validation set: (1241, 7, 14)
Test set:       (1241, 7, 14)

All tensors saved to the 'data' folder. Preprocessing complete!
